# France GTFS — download, merge, tiers

Builds the input of the raptor-kmp benchmark: every feed of the catalogue
(`catalog/gtfs_france.csv`), downloaded at its latest version, normalized, then merged
into **22 métropoles → 13 regions → mainland France**.

Run it top to bottom with the benchmark's environment as kernel (`benchmark/.venv`, see
`benchmark/README.md`). A rerun is cheap: unchanged feeds are not downloaded again,
and normalized feeds are reused while their source, the window and the filters stay
the same.

| Writes | |
|---|---|
| `data/merged/{metropole,region,france}/*.zip` | one GTFS per tier, for the pipeline (local, not committed) |
| `results/reference_day.json`, `coverage.csv`, `coverage.png` | how the reference day was chosen |
| `results/feeds_used.csv` | the exact version of every feed that went in |
| `results/merge_stats.csv`, `tiers.png` | the size of every tier |
| `results/pattern_overlaps.csv` | feeds that run the same stop sequences |
| `data/raptor/<tier>/bench/*.bin`, `results/pipeline.csv` | the pipeline's output for each tier, and what converting it cost |
| `results/engine*.csv`, `results/engine/` | raptor-kmp on every tier (section 11) |
| `results/connected_france.csv`, `connectivity_islands.csv`, `reachability_prefectures.csv`, `reachability.png` | France in one piece (section 13) |

In [ ]:
import datetime as dt
import json
import sys
from pathlib import Path

BENCHMARK_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(BENCHMARK_DIR))

import pandas as pd
from IPython.display import Image, display

from gtfs_bench import charts
from gtfs_bench.catalog import METROPOLES, read_catalog
from gtfs_bench.credentials import CredentialStore, variable_names
from gtfs_bench.download import download_all, read_manifest, summarize
from gtfs_bench.merge import MergeParams, TransferParams, merge_tier, pattern_overlaps, tiers_from_catalog
from gtfs_bench.normalize import Window, connect, normalize_all, trips_per_day

RESULTS_DIR = BENCHMARK_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)
pd.set_option("display.max_rows", 60)
pd.set_option("display.width", 160)

## 1. Configuration

In [ ]:
LAYERS = ("regional",)          # the tiers; the long-distance layer joins in section 13

DOWNLOAD = True                 # False: work from what is already in data/raw
DOWNLOAD_WORKERS = 6
FORCE_DOWNLOAD = False          # True: ignore ETags, fetch everything again

# The reference day is the Tuesday, outside school holidays, that the most feeds run on.
# Set REFERENCE_DATE to a date to impose one instead.
REFERENCE_DATE = None
COVERAGE_DAYS = 42              # how far ahead of today to look for it
SCHOOL_HOLIDAYS = [             # all zones; add a zone's own breaks if the window reaches them
    (dt.date(2026, 12, 19), dt.date(2027, 1, 4)),   # Noël
]

MERGE_PARAMS = dict(
    name_radius_m=500.0,        # same-name stops this close (chained) are one place
    transfers=TransferParams(radius_m=300.0, max_neighbors=30, walk_speed_mps=1.33, floor_s=60),
    memory_limit="6GB",         # DuckDB; this machine has 15 GB
)
TIERS = None                    # None: all of them; or e.g. ["metropole:lyon", "region:84"]

RUN_PIPELINE = True             # convert every merged tier with the pipeline (./gradlew :cli:installDist first)
PIPELINE_XMX = "12g"            # JVM heap ceiling for the pipeline
JAVA = None                     # java executable; None: $JAVA_HOME, then PATH (JDK 21, see README)

# Settings edited in the console (python -m gtfs_bench.console, tab "Paramètres", saved to
# benchmark/settings.json) take over from the values above; delete the file to go back.
from gtfs_bench.settings import load as load_settings
SETTINGS = load_settings()
REFERENCE_DATE = SETTINGS.reference_date() or REFERENCE_DATE
COVERAGE_DAYS = SETTINGS.data.coverage_days
DOWNLOAD_WORKERS = SETTINGS.data.download_workers
MERGE_PARAMS = dict(name_radius_m=SETTINGS.merge.name_radius_m, transfers=SETTINGS.merge.transfer_params(),
                    memory_limit=SETTINGS.merge.duckdb_memory)
PIPELINE_XMX = SETTINGS.engine.pipeline_xmx

## 2. Credentials

Feeds whose catalogue row has an `auth` key need an account. The values are looked up in
`benchmark/.env`, then in the environment. If they are missing, this cell asks for them
(typed input stays hidden). Nothing is printed but whether each key resolved.

A feed registered by hand (`python -m gtfs_bench.download --register <feed_id>`) is not
downloaded again, so it needs no credentials.

In [ ]:
catalog = read_catalog()
rows = [r for r in catalog if r["include"] and r["layer"] in LAYERS]
manifest = read_manifest()
store = CredentialStore()

needed = sorted({r["auth"] for r in rows if r["auth"]
                 and not (r["feed_id"] in manifest and manifest[r["feed_id"]].status == "manual")})
for key in needed:
    if not store.has(key):
        user_var, password_var = variable_names(key)
        try:
            from getpass import getpass
            store.set(key, input(f"{key} — {user_var}: ").strip(), getpass(f"{key} — {password_var}: "))
        except Exception:
            print(f"{key}: no credentials and no way to ask for them here; its feeds will fail to download")
for key in needed:
    print(f"{key}: {'ok' if store.has(key) else 'MISSING'}")
if not needed:
    print("no feed to download needs credentials")

## 3. Catalogue

In [ ]:
frame = pd.DataFrame(rows)
by_region = (frame.groupby(["region_code", "region_name"])
             .agg(feeds=("feed_id", "size"), with_metropole=("metropole", lambda m: (m != "").sum()),
                  levels=("level", lambda l: ", ".join(f"{k} {v}" for k, v in l.value_counts().items())))
             .reset_index())
print(f"{len(rows)} feeds included out of {len(catalog)} catalogue rows, layers {LAYERS}")
by_region

In [ ]:
missing = [key for key in METROPOLES if not any(r["metropole"] == key for r in rows)]
print("métropoles without a feed:", missing or "none")

## 4. Download

In [ ]:
if DOWNLOAD:
    def progress(line, _seen=[0]):
        _seen[0] += 1
        if "failed" in line or _seen[0] % 25 == 0:
            print(line)
    results = download_all(rows, store, workers=DOWNLOAD_WORKERS, force=FORCE_DOWNLOAD, progress=progress)
    print(summarize(results))
manifest = read_manifest()
failed = [r["feed_id"] for r in rows if r["feed_id"] not in manifest or not manifest[r["feed_id"]].ok]
print("feeds unavailable for the merge:", failed or "none")
rows = [r for r in rows if r["feed_id"] not in failed]

## 5. Reference day

The pipeline converts a single day (one profile period). Every feed counts its trips per
date over the coming weeks, from its raw calendars. The day chosen is the Tuesday outside
school holidays that the most feeds have service on, then the one with the most trips.
The merge keeps that day's whole week (Monday to Sunday).

In [ ]:
start = dt.date.today()
coverage_window = Window(start, start + dt.timedelta(days=COVERAGE_DAYS))
con = connect(MERGE_PARAMS["memory_limit"])
per_feed = []
for row in rows:
    days = trips_per_day(row, coverage_window, con)
    days["feed_id"] = row["feed_id"]
    per_feed.append(days)
con.close()
per_feed = pd.concat(per_feed)
per_feed["date"] = pd.to_datetime(per_feed["date"]).dt.date
all_days = pd.DataFrame({"date": [coverage_window.start + dt.timedelta(days=i)
                                  for i in range((coverage_window.end - coverage_window.start).days + 1)]})
daily = (per_feed.groupby("date").agg(feeds=("feed_id", "nunique"), trips=("trips", "sum")).reset_index())
daily = all_days.merge(daily, on="date", how="left").fillna({"feeds": 0, "trips": 0}).astype({"feeds": int, "trips": int})

def in_holidays(day):
    return any(a <= day <= b for a, b in SCHOOL_HOLIDAYS)

candidates = daily[[d.weekday() == 1 and not in_holidays(d) for d in daily["date"]]]
chosen = candidates.sort_values(["feeds", "trips", "date"], ascending=[False, False, True]).iloc[0]
reference = REFERENCE_DATE or chosen["date"]
window = Window.week_of(reference)
not_running = sorted(set(r["feed_id"] for r in rows) - set(per_feed.loc[per_feed["date"] == reference, "feed_id"]))

reference_day = {
    "reference_date": reference.isoformat(), "chosen_by": "config" if REFERENCE_DATE else "coverage",
    "window": window.as_dict(), "feeds_total": len(rows),
    "feeds_running_that_day": int(daily.loc[daily["date"] == reference, "feeds"].iloc[0]),
    "feeds_not_running_that_day": not_running, "school_holidays": [[a.isoformat(), b.isoformat()] for a, b in SCHOOL_HOLIDAYS],
    "computed_on": dt.date.today().isoformat(),
}
(RESULTS_DIR / "reference_day.json").write_text(json.dumps(reference_day, ensure_ascii=False, indent=1), encoding="utf-8")
daily.to_csv(RESULTS_DIR / "coverage.csv", index=False)
print(f"reference day: {reference:%A %d %B %Y}, window {window.start} → {window.end}")
print(f"{reference_day['feeds_running_that_day']} of {len(rows)} feeds run that day; not running: {not_running}")
candidates.set_index("date")

In [ ]:
display(Image(charts.coverage_chart(daily, SCHOOL_HOLIDAYS, reference, RESULTS_DIR / "coverage.png", len(rows))))

## 6. Normalize every feed

One feed at a time, into `data/normalized/<feed_id>/`. The table counts what each feed
lost and why. A feed that fails is reported and left out; the others go on.

In [ ]:
reports, errors = normalize_all(rows, window, {k: m.sha256 for k, m in manifest.items()},
                                memory_limit=MERGE_PARAMS["memory_limit"], progress=lambda line: None)
normalized = pd.DataFrame([r.__dict__ for r in reports]).set_index("feed_id")
print(f"{len(reports)} feeds normalized, {len(errors)} failed")
for feed_id, error in errors.items():
    print("  FAILED", feed_id, error[:300])
rows = [r for r in rows if r["feed_id"] not in errors]
drops = ["stops_not_physical", "stops_outside_france", "trips_dropped_by_agency", "trips_dropped_by_stop_filter",
         "trips_without_service_in_window", "trips_expanded_from_frequencies", "stop_times_interpolated",
         "trips_without_end_times", "trips_too_short"]
print("totals over the window:", normalized[["trips_out", "stop_times_out", "stops_out"]].sum().to_dict())
normalized[drops].sum().to_frame("total")

In [ ]:
print("feeds with no trip in the window:", list(normalized.index[normalized["trips_out"] == 0]))
filters = normalized[normalized["agency_filter_matches"].notna()][["agency_filter_matches", "agencies_dropped"]]
filters

In [ ]:
manifest = read_manifest()
feeds_used = pd.DataFrame([{
    "feed_id": r["feed_id"], "status": manifest[r["feed_id"]].status, "sha256": manifest[r["feed_id"]].sha256,
    "downloaded_at": manifest[r["feed_id"]].downloaded_at, "service_start": manifest[r["feed_id"]].service_start,
    "service_end": manifest[r["feed_id"]].service_end, "trips_in_window": int(normalized.loc[r["feed_id"], "trips_out"]),
    "stop_times_in_window": int(normalized.loc[r["feed_id"], "stop_times_out"]),
} for r in rows])
feeds_used.to_csv(RESULTS_DIR / "feeds_used.csv", index=False)
print(f"{len(feeds_used)} feed versions recorded in results/feeds_used.csv")

## 7. Merge: métropoles → regions → France

One GTFS zip per tier in `data/merged/`. On top of the union of its feeds, each merge:

- clips national feeds to the tier's area;
- drops trips carried twice;
- renames distant homonym stops;
- adds walking transfers;
- checks every reference before writing.

In [ ]:
params = MergeParams(reference_date=reference, **MERGE_PARAMS)
tiers = tiers_from_catalog(rows, LAYERS)
selected = [t for key, t in tiers.items() if TIERS is None or key in TIERS]
tier_reports = {}
for kind in ("metropole", "region", "france"):
    for tier in (t for t in selected if t.kind == kind):
        tier_reports[tier.key] = merge_tier(tier, params)

## 8. Tier statistics

Counts over the merged week, then **on the reference day**: that day is what the
pipeline converts and the engine loads.

In [ ]:
stats = pd.DataFrame([{
    "tier": key, "kind": tiers[key].kind, "name": tiers[key].name, "feeds": len(r.feeds),
    "stops_on_reference_date": r.stops_on_reference_date, "trips_on_reference_date": r.trips_on_reference_date,
    "stop_times_on_reference_date": r.stop_times_on_reference_date,
    "stops": r.stops, "routes": r.routes, "trips": r.trips, "stop_times": r.stop_times,
    "transfers": r.transfers, "transfers_generated": r.transfers_generated,
    "duplicate_trips": r.duplicate_trips, "trips_clipped_away": r.trips_clipped_away,
    "renamed_stops": r.names["renamed_stops"], "implicit_pairs_before": r.names["implicit_pairs_before"],
    "implicit_pairs_after": r.names["implicit_pairs_after"], "largest_name_group_before": r.names["largest_group_before"],
    "zip_mb": round(r.zip_bytes / 1024**2, 1), "merge_seconds": r.seconds, "integrity_problems": len(r.integrity_problems),
} for key, r in tier_reports.items()])
stats.to_csv(RESULTS_DIR / "merge_stats.csv", index=False)
stats[["tier", "name", "feeds", "stops_on_reference_date", "trips_on_reference_date", "stop_times_on_reference_date",
       "transfers", "renamed_stops", "implicit_pairs_before", "implicit_pairs_after", "zip_mb", "merge_seconds"]]

In [ ]:
labelled = {  # the Grand Paris tier is IDFM, as is Île-de-France: one dot under the other, one label
    "metropole:lyon": "Lyon", "metropole:dijon": "Dijon", "region:11": "Île-de-France (= Grand Paris)",
    "region:84": "Auvergne-Rhône-Alpes", "region:94": "Corse", "france": "France",
}
display(Image(charts.tiers_chart(stats, RESULTS_DIR / "tiers.png", labelled=labelled)))

## 9. Checks

These should all come out empty or zero:

- every written tier resolves all of its references;
- no trip is dropped as carried twice by two *different* feeds;
- no two feeds run the same stop sequences. A network present in two versions of its
  timetable slips past trip deduplication, which compares times, so this check compares
  stop sequences only.

In [ ]:
problems = {key: r.integrity_problems for key, r in tier_reports.items() if r.integrity_problems}
print("integrity problems:", problems or "none")
cross = {key: {pair: n for pair, n in r.duplicate_trips_by_feed.items() if pair.split(" -> ")[0] != pair.split(" -> ")[1]}
         for key, r in tier_reports.items()}
print("trips carried by two feeds:", {k: v for k, v in cross.items() if v} or "none")
overlaps = pattern_overlaps([r["feed_id"] for r in rows], memory_limit=MERGE_PARAMS["memory_limit"])
overlaps.to_csv(RESULTS_DIR / "pattern_overlaps.csv", index=False)
overlaps

## 10. Conversion by the pipeline

Every merged tier goes through the pipeline's CLI, as any network does, with a
one-period profile: the reference day. The table shows, per tier:

- time and peak memory (the JVM's peak working set: heap and everything around it);
- the `.bin` sizes;
- how many of the day's trips the pipeline kept. It keeps one stop sequence per route
  and direction, and drops the trips that do not cover it.

Output: `data/raptor/<tier>/bench/` (local) and `results/pipeline.csv`.

In [ ]:
from gtfs_bench.convert import convert_all, write_results

if RUN_PIPELINE:
    conversions = convert_all(reference, list(tier_reports), xmx=PIPELINE_XMX, java=JAVA)
    write_results(conversions)
pipeline = pd.read_csv(RESULTS_DIR / "pipeline.csv")
pipeline[["tier", "status", "seconds", "peak_memory_mb", "input_zip_mb", "stops_bin_mb", "routes_bin_mb",
          "stops", "trips", "trips_found", "trips_kept_share", "stop_times", "transfers", "error"]]

In [ ]:
points = pipeline[pipeline["status"] == "ok"][["tier", "peak_memory_mb"]].merge(stats[["tier", "kind", "stop_times"]], on="tier")
labelled = {"metropole:lyon": "Lyon", "region:11": "Île-de-France", "region:84": "Auvergne-Rhône-Alpes", "france": "France"}
display(Image(charts.pipeline_chart(points, RESULTS_DIR / "pipeline.png", labelled=labelled)))

## 11. The engine: raptor-kmp on every tier

The `:benchmark-engine` harness (`./gradlew :benchmark-engine:installDist`) loads each
tier's `stops.bin` / `routes.bin` into `eu.dotshell:raptor-kmp-jvm`, one fresh JVM per tier.
It measures:

- cold load;
- the heap the loaded network holds;
- latency of 1,000 seeded random trips;
- the same Lyon and Paris trips replayed on every tier that covers them;
- named trips, from Bellecour → Part-Dieu to Brest → Strasbourg.

A sweep then runs Lyon, Auvergne-Rhône-Alpes, Île-de-France and France under
-Xmx 512m, 1g, 2g and 4g, to see where each one stops fitting.

In [ ]:
from gtfs_bench.engine import heap_sweep, run_all, write_tables

RUN_ENGINE = True
if RUN_ENGINE:
    run_all(list(tier_reports), xmx="12g", java=JAVA)
    heap_sweep(java=JAVA)
engine = pd.read_csv(RESULTS_DIR / "engine.csv")
engine["kind"] = engine["tier"].str.split(":").str[0]
summary = engine.pivot_table(index="tier", columns=["query_set", "direction"], values="latency_p95_ms").round(2)
sizes = engine.drop_duplicates("tier").set_index("tier")[["stops", "trips", "bins_mb", "cold_build_ms", "retained_heap_mb"]].round(1)
sizes.join(summary.set_axis([f"p95 {a} {b}" for a, b in summary.columns], axis=1))

In [ ]:
pd.read_csv(RESULTS_DIR / "engine_scenarios.csv")[["tier", "name", "found", "latencyMs", "arrival", "durationMin", "transitLegs", "routes", "note"]]

In [ ]:
pd.read_csv(RESULTS_DIR / "engine_heap.csv")

In [ ]:
kind_of = lambda t: "france" if t == "france" else t.split(":")[0]
local = engine[engine["query_set"].isin(["lyon-local", "paris-local"]) & (engine["direction"] == "forward") & (engine["queries"] > 0)].copy()
local = local[~((local["query_set"] == "paris-local") & (local["tier"] == "region:11"))]   # IDFM twice: keep one
local["kind"] = local["tier"].map(kind_of)
local["label"] = local["tier"].map({"metropole:lyon": "Lyon", "region:84": "AURA", "metropole:grand-paris": "Grand Paris / IDF", "france": "France"})
local = local.sort_values(["query_set", "stops"])
display(Image(charts.local_trips_chart(local, RESULTS_DIR / "engine_local_trips.png")))

points = engine[(engine["query_set"] == "random") & (engine["direction"] == "forward")].copy()
points["kind"] = points["tier"].map(kind_of)
labelled = {"metropole:lyon": "Lyon", "region:11": ("Île-de-France", -9, 4), "region:84": ("Auvergne-Rhône-Alpes", 7, -12), "france": "France"}
display(Image(charts.scaling_chart(points, "stops", "retained_heap_mb", "Stops in the network", "Heap held by the loaded network",
                                   "Engine memory grows with the network", RESULTS_DIR / "engine_heap.png", labelled, "MB")))
display(Image(charts.scaling_chart(points, "stops", "latency_p50_ms", "Stops in the network", "Median time of a random trip",
                                   "Query time grows with the network", RESULTS_DIR / "engine_latency.png", labelled, "ms")))

## 12. Same-name links (optional)

raptor-kmp links every pair of stops sharing a name with a 120 s transfer. The merge renames
homonyms by place, so this cell measures what happens without that: France merged with
the producers' names, converted and timed like the real tier. The result goes to
`results/engine_same_name_links.csv` (see RESULTS.md). It reconverts France, so it is off
by default.

In [ ]:
RUN_SAME_NAME_VARIANT = False
if RUN_SAME_NAME_VARIANT:
    from gtfs_bench.variants import same_name_variant
    same_name_variant(reference, java=JAVA)
path = RESULTS_DIR / "engine_same_name_links.csv"
pd.read_csv(path) if path.exists() else "not run yet"

## 13. France connexe: the long-distance layer

The France tier above is the regions side by side, linked by TER and walks only. This
section builds two variants of it, each changing one thing, and puts all three through
the same chain (merge, pipeline, engine) plus two checks of the step:

| Network | What changes |
|---|---|
| `france` | the tier above (France juxtaposée) |
| `france-patterns` | every route split into one route per stop sequence: the pipeline keeps one sequence per route and direction and drops the trips that skip one of its stops, so split, it keeps them all |
| `france-connexe` | `france-patterns` + the catalogue's `longue_distance` rows (TGV, OUIGO, Intercités, Eurostar, Trenitalia, Renfe, FlixBus, BlaBlaCar Bus, ferries) + wider links around the stations of national networks (500 m, 60 neighbours, and SNCF UIC codes) |
| `france-complete` | france-connexe plus the catalogue's `scolaire` layer: the feeds that carry only school transport (step 10) |

The checks:

- **one piece?** connected components of the reference day's graph (rides and walks), and
  the islands left with their likely cause;
- **from Paris?** raptor-kmp searches every préfecture from Châtelet at 07:00, 8 rounds.

The engine runs at the default 5 rounds and at 8. It takes about 20 minutes (two merges and
two pipeline runs of France), so it can be switched off to show the last results.

In [ ]:
from gtfs_bench.connected import VARIANTS, run
from gtfs_bench.connectivity import outlines

RUN_CONNECTED = True
if RUN_CONNECTED:
    run(reference, java=JAVA)
connected = pd.read_csv(RESULTS_DIR / "connected_france.csv").set_index("network")
connected.T

In [ ]:
reach = pd.read_csv(RESULTS_DIR / "reachability_prefectures.csv")
titles = {v.key: v.label for v in VARIANTS if v.key in set(reach["network"])}
display(Image(charts.reachability_map(reach, outlines(), titles, reference, RESULTS_DIR / "reachability.png")))
reach.pivot_table(index=["department", "prefecture"], columns="network", values="duration_min").sort_values("france-connexe" if "france-connexe" in titles else list(titles)[-1])

In [ ]:
islands = pd.read_csv(RESULTS_DIR / "connectivity_islands.csv")
print("islands of 10 stops or more:", connected["islands_10_plus"].to_dict())
islands[islands["stops"] >= 10][["network", "stops", "feed", "region", "example_stop", "gap_to_main_m", "closest_main_stop", "likely_cause"]]

In [ ]:
scenarios = pd.read_csv(RESULTS_DIR / "connected_scenarios.csv")
scenarios[scenarios["kind"] == "long-distance"][["network", "max_rounds", "name", "found", "latencyMs", "arrival", "durationMin", "transitLegs", "routes", "note"]]

## 14. School lines

Every converted variant gets its school lines (`gtfs_bench/school.py`): the catalogue's `school` column, the
GTFS school route types 712/713, the name, or a line serving a school with at most 6 runs that day. They go to
`school_routes.json` next to the `.bin` files, as the route numbers the engine knows, and the explorer's
"lignes scolaires" switch leaves them out. Below: per feed, then the same trips timed with and without them.

In [ ]:
school_feeds = pd.read_csv(RESULTS_DIR / "school_feeds.csv")
print(f"{school_feeds['school_lines'].sum()} school lines in {len(school_feeds)} feeds, "
      f"{school_feeds['school_trips'].sum()} trips on the reference day")
display(school_feeds.head(20))
switch = pd.read_csv(RESULTS_DIR / "school_switch.csv")
switch.pivot_table(index=["network", "query_set", "direction"], columns="lines",
                   values=["found", "latency_p50_ms", "latency_p95_ms"])

## 15. Live network: horizon, day types, daily refresh

raptor-kmp has no calendar: one network is one day's timetable. The live network covers every date the
feeds still publish, with one network per day type (`gtfs_bench/horizon.py`):

- **window:** from today to the day before the first feed that matters stops (more than 0.5 % of the trips,
  or the one that takes the small feeds already stopped past 1 %); the small ones that stop earlier are listed;
- **day types:** a Tuesday, a Saturday and a Sunday of the earliest week ahead, plus a holiday Tuesday when the
  window reaches national school holidays; a date runs on its day type, public holidays as Sundays.

Producers publish a few weeks ahead and replace their files as they go, so the window only stays ahead if it
is rebuilt: `python -m gtfs_bench.refresh` (or `refresh.ps1`, for the task scheduler) downloads what changed,
reports newer PAN resources of feeds about to stop, recomputes the window and rebuilds
`data/raptor/france-live` when needed. The explorer opens it by default.

In [ ]:
from gtfs_bench import horizon, refresh

RUN_REFRESH = False   # True: the daily refresh from here (downloads, then rebuilds france-live if needed)
if RUN_REFRESH:
    refresh.refresh(java=JAVA)
plan, feed_table, feed_days = horizon.plan(horizon.included_rows(), dt.date.today())
horizon.write_results(plan, feed_table, feed_days)
display(Image(RESULTS_DIR / "feed_horizon.png"))
print(f"window {plan.start} → {plan.end}, set by {plan.binding_feed} ({plan.binding_share:.1%} of the trips)")
print("day types:", plan.days)
pd.DataFrame(plan.lost)

In [ ]:
last = refresh.LIVE_DIR / "refresh.json"
json.loads(last.read_text(encoding="utf-8")) if last.exists() else "no refresh yet"